# BB84 Complete Pipeline — Weeks 2 through 5

This notebook combines the corrected BB84 implementation with:

- **Week 2:** Qiskit Aer BB84 simulation with optional intercept-resend Eve
- **Week 3:** depolarizing, readout, amplitude-damping, and combined noise
- **Week 3:** graphs showing how noise changes QBER
- **Week 4:** hundreds of complete BB84 simulations and CSV generation
- **Week 5:** Random Forest noise-severity classification and evaluation metrics

The machine-learning model avoids the original target leakage. It does not use
`alice_basis` and `bob_basis` to predict a target defined directly from those
same columns.

## 1. Imports and project paths

In [ ]:
from pathlib import Path
import sys

import pandas as pd
from IPython.display import display, Image

CURRENT_DIRECTORY = Path.cwd()
PROJECT_ROOT = (
    CURRENT_DIRECTORY.parent
    if CURRENT_DIRECTORY.name == "notebooks"
    else CURRENT_DIRECTORY
)
SRC_DIRECTORY = PROJECT_ROOT / "src"

if str(SRC_DIRECTORY) not in sys.path:
    sys.path.insert(0, str(SRC_DIRECTORY))

from bb84.core import (
    build_prepare_measure_circuit,
    simulate_bb84_session,
)
from bb84.pipeline import (
    ExperimentConfig,
    generate_experiment_dataset,
    plot_experiment_results,
    train_random_forest,
)

DATASET_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "bb84_experiment_dataset.csv"
)
FIGURES_DIRECTORY = PROJECT_ROOT / "results" / "figures"
MODELS_DIRECTORY = PROJECT_ROOT / "results" / "models"
METRICS_DIRECTORY = PROJECT_ROOT / "results" / "metrics"

print(f"Project root: {PROJECT_ROOT}")
print("Imports loaded successfully.")

## 2. Week 2 — Build and inspect a BB84 circuit

In [ ]:
example_circuit = build_prepare_measure_circuit(
    prepared_bit=1,
    preparation_basis=1,
    measurement_basis=1,
)

print(example_circuit)
example_circuit.draw("mpl")

## 3. Run one ideal BB84 session

In [ ]:
ideal_session = simulate_bb84_session(
    transmissions=256,
    eve_present=False,
    noise_type="ideal",
    noise_strength=0.0,
    seed=42,
)

pd.Series(ideal_session, name="ideal_session")

## 4. Run one session with Eve

In [ ]:
eve_session = simulate_bb84_session(
    transmissions=256,
    eve_present=True,
    noise_type="ideal",
    noise_strength=0.0,
    seed=43,
)

pd.DataFrame(
    [ideal_session, eve_session],
    index=["Eve absent", "Eve present"],
)[
    [
        "eve_present",
        "sifted_key_length",
        "key_accuracy",
        "qber",
    ]
]

## 5. Week 3 — Test each Qiskit Aer noise model

In [ ]:
noise_examples = []

for noise_type in [
    "ideal",
    "depolarizing",
    "readout",
    "amplitude_damping",
    "combined",
]:
    noise_examples.append(
        simulate_bb84_session(
            transmissions=256,
            eve_present=False,
            noise_type=noise_type,
            noise_strength=(
                0.0 if noise_type == "ideal" else 0.08
            ),
            seed=100 + len(noise_examples),
        )
    )

noise_example_df = pd.DataFrame(noise_examples)
display(
    noise_example_df[
        [
            "noise_type",
            "noise_strength",
            "qber",
            "qber_z_basis",
            "qber_x_basis",
            "zero_to_one_error_rate",
            "one_to_zero_error_rate",
        ]
    ]
)

## 6. Week 4 — Generate the full CSV dataset

The default configuration runs **220 complete BB84 sessions**:

- 20 ideal sessions
- 200 noisy sessions
- 4 noise models
- 5 nonzero noise strengths
- Eve present and absent
- 128 transmitted qubits per session

This can take several minutes. Each output row represents one complete BB84
session rather than one individual qubit.

In [ ]:
config = ExperimentConfig()
config

In [ ]:
experiment_df = generate_experiment_dataset(
    config,
    output_path=DATASET_PATH,
    print_progress=True,
)

print(f"Dataset rows: {len(experiment_df)}")
print(f"Dataset columns: {len(experiment_df.columns)}")
print(f"Saved to: {DATASET_PATH}")
display(experiment_df.head(20))

## 7. Inspect the generated dataset

In [ ]:
display(
    experiment_df.groupby(
        ["noise_class", "eve_present"]
    )["qber"].agg(["count", "mean", "std"])
)

## 8. Week 3 graphs — Effect of noise

In [ ]:
noise_figure_paths = plot_experiment_results(
    experiment_df,
    FIGURES_DIRECTORY,
)

for path in noise_figure_paths:
    print(path)
    display(Image(filename=str(path)))

## 9. Week 5 — Train the Random Forest

Target:

- `ideal`
- `low`
- `medium`
- `high`

Excluded from the feature matrix:

- true `noise_strength`
- true `noise_type`
- target `noise_class`
- `run_id`
- `repetition`

This avoids giving the model the answer.

In [ ]:
rf_metrics = train_random_forest(
    experiment_df,
    MODELS_DIRECTORY,
    METRICS_DIRECTORY,
    FIGURES_DIRECTORY,
    random_state=42,
)

print(f"Accuracy: {rf_metrics['accuracy']:.4f}")
print(
    "Balanced accuracy: "
    f"{rf_metrics['balanced_accuracy']:.4f}"
)
print(f"Macro F1: {rf_metrics['macro_f1']:.4f}")
print(f"Training rows: {rf_metrics['training_rows']}")
print(f"Testing rows: {rf_metrics['testing_rows']}")

## 10. Prediction metrics

In [ ]:
classification_report_df = pd.DataFrame(
    rf_metrics["classification_report"]
).transpose()

display(classification_report_df)

## 11. Confusion matrix and feature importance

In [ ]:
display(Image(filename=rf_metrics["confusion_figure"]))
display(Image(filename=rf_metrics["importance_figure"]))

## 12. Final output files

In [ ]:
output_files = {
    "dataset": DATASET_PATH,
    "model": Path(rf_metrics["model_path"]),
    "metrics": Path(rf_metrics["metrics_path"]),
    "classification_report": (
        METRICS_DIRECTORY / "classification_report.csv"
    ),
    "feature_importance": (
        METRICS_DIRECTORY / "feature_importance.csv"
    ),
    "qber_noise_graph": (
        FIGURES_DIRECTORY / "qber_vs_noise.png"
    ),
    "eve_comparison_graph": (
        FIGURES_DIRECTORY / "qber_eve_comparison.png"
    ),
    "damping_graph": (
        FIGURES_DIRECTORY
        / "amplitude_damping_directionality.png"
    ),
    "confusion_matrix": Path(
        rf_metrics["confusion_figure"]
    ),
    "model_feature_importance": Path(
        rf_metrics["importance_figure"]
    ),
}

for name, path in output_files.items():
    print(f"{name}: {path}")